In [ ]:
pip install sentence-transformers faiss-cpu numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 51.2 MB/s eta 0:00:00


In [ ]:
knowledge_base = [
    "You can reset your password using the Forgot Password option.",
    "Billing invoices can be downloaded from the billing section.",
    "Users can update their profile information from account settings.",
    "If you cannot log in, check your username and password.",
    "Two-factor authentication improves account security.",
    "Subscription plans can be upgraded anytime from the dashboard.",
    "Refund requests are processed within five business days.",
    "Your account can be deactivated from the privacy settings page.",
    "Payment methods can be added or removed in billing settings.",
    "Login issues may occur if your account is temporarily locked."
]

print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings
embeddings = model.encode(
    knowledge_base,
    convert_to_numpy=True
)

print("\nEmbedding Matrix Shape:")
print(embeddings.shape)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Embedding Matrix Shape:
(10, 384)


In [ ]:
dimension = embeddings.shape[1]

# Normalize embeddings for cosine similarity behavior
faiss.normalize_L2(embeddings)

# Create index
index = faiss.IndexFlatL2(dimension)

# Add vectors
index.add(embeddings)

print("\nFAISS Index Created")
print("Total vectors stored:", index.ntotal)


FAISS Index Created
Total vectors stored: 10


In [ ]:
def semantic_search(query, top_k=3):

    # Encode query
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    # Normalize query vector
    faiss.normalize_L2(query_embedding)

    # Search
    distances, indices = index.search(query_embedding, top_k)

    print("\nQuery:", query)
    print("-" * 80)
    print(f"{'Rank':<5} {'Score':<15} Matched Sentence")
    print("-" * 80)

    for rank, (idx, score) in enumerate(
        zip(indices[0], distances[0]),
        start=1
    ):
        print(
            f"{rank:<5} {score:<15.4f} {knowledge_base[idx]}"
        )


In [ ]:

test_queries = [
    "How do I change my password?",
    "Where can I see my invoices?",
    "I am unable to sign into my account"
]

print("\n\n========= TEST SEARCHES =========")

for q in test_queries:
    semantic_search(q)



========= TEST SEARCHES =========

Query: How do I change my password?
--------------------------------------------------------------------------------
Rank  Score           Matched Sentence
--------------------------------------------------------------------------------
1     0.6911          You can reset your password using the Forgot Password option.
2     1.1456          If you cannot log in, check your username and password.
3     1.2841          Your account can be deactivated from the privacy settings page.

Query: Where can I see my invoices?
--------------------------------------------------------------------------------
Rank  Score           Matched Sentence
--------------------------------------------------------------------------------
1     0.5542          Billing invoices can be downloaded from the billing section.
2     1.3329          Payment methods can be added or removed in billing settings.
3     1.5516          Refund requests are processed within five business d

In [ ]:
print("\n\n========= INTERACTIVE SEARCH =========")
print("Type 'exit' to quit.")

while True:

    user_query = input("\nEnter query: ")

    if user_query.lower() == "exit":
        print("Goodbye!")
        break

    semantic_search(user_query)



========= INTERACTIVE SEARCH =========
Type 'exit' to quit.

Enter query: exit
Goodbye!
